# Similarity Analysis

In [ ]:
from pathlib import Path
import pandas as pd
from src.Similarity.LoadData import SimilarityDataLoader
from src.Similarity.TemporalAggregation import SimilarityTemporalAggregator
from src.Similarity.Preprocessing import SimilarityPreprocessor
from src.Similarity.SimilarityAnalyzer import SimilarityAnalyzer
from src.Plots.SimilarityDistanceHeatmapPlot import SimilarityDistanceHeatmapPlot
from src.Plots.SimilarityDendrogramPlot import SimilarityDendrogramPlot
from src.Similarity.StreamBuilder import SimilarityStreamBuilder
from src.Similarity.OneVsAll import SimilarityOneVsAll
from src.Similarity.Experiments import SimilarityExperimentRunner

## 1. Create Temporal Cuts

Cria recortes temporais dos CSVs originais para limitar o período analisado de cada ataque e reutilizar os dados nas etapas seguintes.

- Normaliza nomes de colunas e rótulos.
- Calcula o término de cada fluxo usando timestamp e duração.
- Extrai o intervalo configurado a partir do primeiro término de fluxo do ataque-alvo.
- Salva os recortes e um relatório com as contagens por tipo de tráfego.


In [ ]:
dataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")

attackFiles = {
    "NTP": "DrDoS_NTP.csv",
    "DNS": "DrDoS_DNS.csv",
    "LDAP": "DrDoS_LDAP.csv",
    "MSSQL": "DrDoS_MSSQL.csv",
    "NETBIOS": "DrDoS_NetBIOS.csv",
    "SNMP": "DrDoS_SNMP.csv",
    "SSDP": "DrDoS_SSDP.csv",
    "UDP": "DrDoS_UDP.csv",
    "UDP-LAG": "UDPLag.csv",
    "SYN": "Syn.csv",
    "TFTP": "TFTP.csv",
}

durationMinutes = 5.0
chunkSize = 100_000
normalLabels = ["BENIGN", "NORMAL", "0"]

cutsDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/cuts")
cutSummaryPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/temporal_cuts.csv")

loader = SimilarityDataLoader(
    normalLabels=normalLabels,
    chunkSize=chunkSize,
)

loader.createCuts(
    dataDirectory=dataDirectory,
    attackFiles=attackFiles,
    durationMinutes=durationMinutes,
    outputDirectory=cutsDirectory,
    summaryPath=cutSummaryPath,
)

display(pd.read_csv(cutSummaryPath))


## 2. Create Datasets

Os recortes originam três representações, permitindo comparar o efeito da presença de tráfego benigno e da agregação temporal na similaridade entre ataques.

- **Attack-Only:** remove benignos antes da agregação. Cada janela representa exclusivamente os fluxos do ataque-alvo; contagens, médias e demais características descrevem somente esse ataque.
- **Mixed Traffic:** preserva os benignos junto ao ataque-alvo. As características são calculadas sobre todo esse tráfego. Uma janela recebe o rótulo do ataque-alvo quando contém pelo menos um fluxo dele. Janelas exclusivamente benignas são excluídas no pré-processamento.
- **Stream:** seleciona as primeiras N amostras do ataque-alvo e ordena o conjunto selecionado pelo timestamp, preservando as características dos fluxos, sem agregação.


### 2.1 Attack-Only

Filtra cada recorte para manter somente os fluxos do ataque antes da agregação.

In [ ]:

cutsDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/cuts")
cutSummaryPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/temporal_cuts.csv")
perAttackDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/windows/attack_only")
combinedOutputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/attack_only_raw_1s.csv")
summaryOutputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/attack_only_window_summary.csv")

windowSeconds = 1
fillEmptyWindows = True
normalLabels = ["BENIGN", "NORMAL", "0"]
noFlowLabel = "NOLABEL"

aggregator = SimilarityTemporalAggregator(
    windowSeconds=windowSeconds,
    normalLabels=normalLabels,
    fillEmptyWindows=fillEmptyWindows,
    noFlowLabel=noFlowLabel,
)

aggregator.aggregateDirectory(
    cutsDirectory=cutsDirectory,
    cutSummaryPath=cutSummaryPath,
    mode="attack_only",
    perAttackDirectory=perAttackDirectory,
    combinedOutputPath=combinedOutputPath,
    summaryOutputPath=summaryOutputPath,
)

print("Dataset bruto Attack-Only:", combinedOutputPath)
display(pd.read_csv(summaryOutputPath))


### 2.2 Mixed Traffic

Mantém somente ataque-alvo e benignos do recorte. Qualquer outro ataque é excluído antes do cálculo das características.


In [ ]:

cutsDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/cuts")
cutSummaryPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/temporal_cuts.csv")
perAttackDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/windows/mixed_traffic")
combinedOutputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/mixed_traffic_raw_1s.csv")
summaryOutputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/mixed_traffic_window_summary.csv")

windowSeconds = 1
fillEmptyWindows = True
normalLabels = ["BENIGN", "NORMAL", "0"]
noFlowLabel = "NOLABEL"

aggregator = SimilarityTemporalAggregator(
    windowSeconds=windowSeconds,
    normalLabels=normalLabels,
    fillEmptyWindows=fillEmptyWindows,
    noFlowLabel=noFlowLabel,
)

aggregator.aggregateDirectory(
    cutsDirectory=cutsDirectory,
    cutSummaryPath=cutSummaryPath,
    mode="mixed_traffic",
    perAttackDirectory=perAttackDirectory,
    combinedOutputPath=combinedOutputPath,
    summaryOutputPath=summaryOutputPath,
)

print("Dataset bruto Mixed Traffic:", combinedOutputPath)
display(pd.read_csv(summaryOutputPath))


### 2.3 Stream

Usa os cuts já existentes. Filtra somente o rótulo do ataque-alvo e pega as primeiras `samplesPerAttack` linhas encontradas, interrompendo a leitura ao atingir a quantidade. Depois reúne as amostras selecionadas e ordena somente esse conjunto pelo timestamp original. Sem sorteio, agregação, novas features ou leitura dos CSVs originais. Se o cut não contiver N amostras do ataque, informa a quantidade disponível sem gerar um Stream desbalanceado.


In [ ]:
samplesPerAttack = 10_000  # Exatamente esta quantidade de registros originais por ataque

streamResult = SimilarityStreamBuilder().runFromFiles(
    dataDirectory=Path(r"E:\CICDDoS2019\01-12\CSV"),
    attackFiles={
        "DNS": "DrDoS_DNS.csv", "UDP": "DrDoS_UDP.csv", "NTP": "DrDoS_NTP.csv",
        "LDAP": "DrDoS_LDAP.csv", "MSSQL": "DrDoS_MSSQL.csv", "NETBIOS": "DrDoS_NetBIOS.csv",
        "SNMP": "DrDoS_SNMP.csv", "SSDP": "DrDoS_SSDP.csv", "UDP-LAG": "UDPLag.csv",
        "SYN": "Syn.csv", "TFTP": "TFTP.csv",
    },
    samplesPerAttack=samplesPerAttack,
    chunkSize=10_000,
    timestampColumn="Timestamp",
    dayFirst=True,  # Datas do CICDDoS2019 no formato dia/mês/ano
    outputPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/stream_raw_flows.csv"),
    summaryOutputPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/reports/stream_flow_selection.csv"),
)
print("Fluxos por ataque:", streamResult["selectedFlowsPerAttack"])
print("Total de fluxos:", streamResult["totalFlows"])
print("CSV:", streamResult["datasetPath"])
display(pd.read_csv(streamResult["summaryPath"]))


## 3. Processing Datasets

O fluxo abaixo se repete para **Attack-Only, Mixed Traffic e Stream**, com processamento independente de cada representação.

- **Preprocessing:** exclui rótulos benignos e vazios, permite downsample e remove colunas sem informação antes da imputação pela mediana global. A remoção por correlação de Pearson e a padronização são opcionais.
- **One-vs-All:** seleciona Top-K características por ataque usando Random Forest e combina as listas por união ou recorrência.
- **Matrix Covariance:** calcula a distância de Mahalanobis entre os centroides dos ataques, usando a covariância dos resíduos internos regularizada por Ledoit-Wolf.
- **Distance Levels:** classifica as distâncias por quantis em semelhantes, intermediárias e distintas, relativamente ao cenário.
- **Heatmap and Dendrogram:** apresenta as distâncias e os agrupamentos entre ataques.


### Attack Only

#### Preprocessing

In [ ]:

datasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/attack_only_raw_1s.csv")

selectedFeatures = None
downsample = "random"  # None preserva todas as janelas
removeZeroColumns = True  # None desativa
removeAllMissingColumns = True  # None desativa
removeConstantColumns = True  # None desativa
imputation = "median"  # Mediana global; None desativa
normalization = "standard"  # None desativa
nearConstantRatio = 0.999
correlationThreshold = 0.995
missingDisplayThreshold = 51.0
randomSeed = 42
noFlowLabel = "NOLABEL"

preprocessor = SimilarityPreprocessor(
    downsample=downsample,
    removeZeroColumns=removeZeroColumns,
    removeAllMissingColumns=removeAllMissingColumns,
    removeConstantColumns=removeConstantColumns,
    imputation=imputation,
    normalization=normalization,
    nearConstantRatio=nearConstantRatio,
    correlationThreshold=correlationThreshold,
    randomSeed=randomSeed,
    missingDisplayThreshold=missingDisplayThreshold,
    noFlowLabel=noFlowLabel,
)

result = preprocessor.runFromFile(
    datasetPath=datasetPath,
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only"),
    selectedFeatures=selectedFeatures,
)

print("Base balanceada:", result["balancedDatasetPath"])
print("Base padronizada:", result["standardizedDatasetPath"])
print("Quantidade de features:", len(result["features"]))

print("Downsample:", downsample)
print("\nAmostras disponíveis, selecionadas e descartadas:")
display(pd.DataFrame.from_dict(result["balanceSummary"], orient="index"))


#### One-vs-All

In [ ]:
extractionMode = "union"  # "union", "recurrence" ou None
featuresPerAttack = 10  # Ex.: 20 para recorrência; também aceita {"DNS": 10, ...}
minAttackOccurrences = 2
selection = SimilarityOneVsAll(
    featuresPerAttack=featuresPerAttack,
    extractionMode=extractionMode,
    minAttackOccurrences=minAttackOccurrences,
    nEstimators=300,
    randomSeed=42,
).runFromFile(
    datasetPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/datasets/imputed_attack_windows.csv"),
    featureSelectionPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/feature_selection.json"),
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/one_vs_all") / (extractionMode or "disabled"),
)
for attack, features in selection["selectedFeaturesByAttack"].items():
    print(attack, features)
print("Seleção final:", selection["selectedFeatures"])
print("Quantidade final:", selection["featureCount"])

#### Matrix Covariance

In [ ]:
standardizedDatasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/datasets/standardized_attack_windows.csv")
featureSelectionPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/feature_selection.json")
reportDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports")

result = SimilarityAnalyzer().runFromFile(
    standardizedDatasetPath=standardizedDatasetPath,
    featureSelectionPath=featureSelectionPath,
    reportDirectory=reportDirectory,
)

display(pd.read_csv(result["distanceMatrixPath"], index_col=0))

#### Distance Levels

In [ ]:
pairwisePath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/pairwise_mahalanobis_distances.csv")
levelsPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/pairwise_distance_levels.csv")

similarQuantile = 0.33
distinctQuantile = 0.67

levelsPath, similarLimit, distinctLimit = SimilarityAnalyzer().classifyDistanceFile(
    pairwisePath=pairwisePath,
    outputPath=levelsPath,
    similarQuantile=similarQuantile,
    distinctQuantile=distinctQuantile,
)

levels = pd.read_csv(levelsPath)
print("Pares semelhantes:")
display(levels.query("DistanceLevel == 'similar'"))

print("Pares intermediários:")
display(levels.query("DistanceLevel == 'intermediate'"))

print("Pares distintos:")
display(levels.query("DistanceLevel == 'distinct'").sort_values("MahalanobisDistance", ascending=False))

#### Heatmap and Dendrogram

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/plots/attack_distance_dendrogram.png")

SimilarityDendrogramPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Dendrograma salvo em:", outputPath)

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/attack_only/plots/mahalanobis_distance_heatmap.png")

SimilarityDistanceHeatmapPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Heatmap salvo em:", outputPath)

### Mixed Traffic

#### Preprocessing

In [ ]:
datasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/mixed_traffic_raw_1s.csv")

selectedFeatures = None
downsample = "random"  # None preserva todas as janelas
removeZeroColumns = True  # None desativa
removeAllMissingColumns = True  # None desativa
removeConstantColumns = True  # None desativa
imputation = "median"  # Mediana global; None desativa
normalization = "standard"  # None desativa
nearConstantRatio = 0.999
correlationThreshold = 0.995
missingDisplayThreshold = 30.0
randomSeed = 42
noFlowLabel = "NOLABEL"

preprocessor = SimilarityPreprocessor(
    downsample=downsample,
    removeZeroColumns=removeZeroColumns,
    removeAllMissingColumns=removeAllMissingColumns,
    removeConstantColumns=removeConstantColumns,
    imputation=imputation,
    normalization=normalization,
    nearConstantRatio=nearConstantRatio,
    correlationThreshold=correlationThreshold,
    randomSeed=randomSeed,
    missingDisplayThreshold=missingDisplayThreshold,
    noFlowLabel=noFlowLabel,
)

result = preprocessor.runFromFile(
    datasetPath=datasetPath,
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic"),
    selectedFeatures=selectedFeatures,
)

print("Base balanceada:", result["balancedDatasetPath"])
print("Base padronizada:", result["standardizedDatasetPath"])
print("Quantidade de features:", len(result["features"]))

print("Downsample:", downsample)
print("\nAmostras disponíveis, selecionadas e descartadas:")
display(pd.DataFrame.from_dict(result["balanceSummary"], orient="index"))


#### One-vs-All

In [ ]:
extractionMode = "union"  # "union", "recurrence" ou None
featuresPerAttack = 10
minAttackOccurrences = 2
selection = SimilarityOneVsAll(
    featuresPerAttack=featuresPerAttack,
    extractionMode=extractionMode,
    minAttackOccurrences=minAttackOccurrences,
    nEstimators=300,
    randomSeed=42,
).runFromFile(
    datasetPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/datasets/imputed_attack_windows.csv"),
    featureSelectionPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/feature_selection.json"),
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/one_vs_all") / (extractionMode or "disabled"),
)
for attack, features in selection["selectedFeaturesByAttack"].items():
    print(attack, features)
print("Seleção final:", selection["selectedFeatures"])
print("Quantidade final:", selection["featureCount"])

#### Matrix Covariance

In [ ]:
standardizedDatasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/datasets/standardized_attack_windows.csv")
featureSelectionPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/feature_selection.json")
reportDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports")

result = SimilarityAnalyzer().runFromFile(
    standardizedDatasetPath=standardizedDatasetPath,
    featureSelectionPath=featureSelectionPath,
    reportDirectory=reportDirectory,
)

display(pd.read_csv(result["distanceMatrixPath"], index_col=0))

#### Distance Levels

In [ ]:
pairwisePath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/pairwise_mahalanobis_distances.csv")
levelsPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/pairwise_distance_levels.csv")

similarQuantile = 0.33
distinctQuantile = 0.67

levelsPath, similarLimit, distinctLimit = SimilarityAnalyzer().classifyDistanceFile(
    pairwisePath=pairwisePath,
    outputPath=levelsPath,
    similarQuantile=similarQuantile,
    distinctQuantile=distinctQuantile,
)

levels = pd.read_csv(levelsPath)
print("Pares semelhantes:")
display(levels.query("DistanceLevel == 'similar'"))

print("Pares intermediários:")
display(levels.query("DistanceLevel == 'intermediate'"))

print("Pares distintos:")
display(levels.query("DistanceLevel == 'distinct'").sort_values("MahalanobisDistance", ascending=False))

#### Heatmap and Dendrogram

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/plots/mahalanobis_distance_heatmap.png")

SimilarityDistanceHeatmapPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Heatmap salvo em:", outputPath)

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/mixed_traffic/plots/attack_distance_dendrogram.png")

SimilarityDendrogramPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Dendrograma salvo em:", outputPath)

### Stream

#### Preprocessing

In [ ]:

datasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/datasets/stream_raw_flows.csv")

selectedFeatures = None
downsample = None  # Preserva todos os fluxos selecionados
removeZeroColumns = True  # None desativa
removeAllMissingColumns = True  # None desativa
removeConstantColumns = True  # None desativa
imputation = "median"  # Mediana global; None desativa
normalization = "standard"  # None desativa
nearConstantRatio = 0.999
correlationThreshold = 0.995
missingDisplayThreshold = 51.0
randomSeed = 42
noFlowLabel = "NOLABEL"

preprocessor = SimilarityPreprocessor(
    sampleUnit="flow",
    downsample=downsample,
    removeZeroColumns=removeZeroColumns,
    removeAllMissingColumns=removeAllMissingColumns,
    removeConstantColumns=removeConstantColumns,
    imputation=imputation,
    normalization=normalization,
    nearConstantRatio=nearConstantRatio,
    correlationThreshold=correlationThreshold,
    randomSeed=randomSeed,
    missingDisplayThreshold=missingDisplayThreshold,
    noFlowLabel=noFlowLabel,
)

result = preprocessor.runFromFile(
    datasetPath=datasetPath,
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream"),
    selectedFeatures=selectedFeatures,
)

print("Fluxos balanceados:", result["balancedDatasetPath"])
print("Base padronizada:", result["standardizedDatasetPath"])
print("Quantidade de features:", len(result["features"]))

print("Downsample:", downsample)
print("\nAmostras disponíveis, selecionadas e descartadas:")
display(pd.DataFrame.from_dict(result["balanceSummary"], orient="index"))


#### One-vs-All

In [ ]:
extractionMode = "union"  # "union", "recurrence" ou None
featuresPerAttack = 10  # Ex.: 20 para recorrência; também aceita {"DNS": 10, ...}
minAttackOccurrences = 2
selection = SimilarityOneVsAll(
    featuresPerAttack=featuresPerAttack,
    extractionMode=extractionMode,
    minAttackOccurrences=minAttackOccurrences,
    nEstimators=300,
    randomSeed=42,
).runFromFile(
    datasetPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/datasets/imputed_attack_flows.csv"),
    featureSelectionPath=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/feature_selection.json"),
    outputDirectory=Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/one_vs_all") / (extractionMode or "disabled"),
)
for attack, features in selection["selectedFeaturesByAttack"].items():
    print(attack, features)
print("Seleção final:", selection["selectedFeatures"])
print("Quantidade final:", selection["featureCount"])

#### Matrix Covariance

In [ ]:
standardizedDatasetPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/datasets/standardized_attack_flows.csv")
featureSelectionPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/feature_selection.json")
reportDirectory = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports")

result = SimilarityAnalyzer().runFromFile(
    standardizedDatasetPath=standardizedDatasetPath,
    featureSelectionPath=featureSelectionPath,
    reportDirectory=reportDirectory,
)

display(pd.read_csv(result["distanceMatrixPath"], index_col=0))

#### Distance Levels

In [ ]:
pairwisePath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/pairwise_mahalanobis_distances.csv")
levelsPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/pairwise_distance_levels.csv")

similarQuantile = 0.33
distinctQuantile = 0.67

levelsPath, similarLimit, distinctLimit = SimilarityAnalyzer().classifyDistanceFile(
    pairwisePath=pairwisePath,
    outputPath=levelsPath,
    similarQuantile=similarQuantile,
    distinctQuantile=distinctQuantile,
)

levels = pd.read_csv(levelsPath)
print("Pares semelhantes:")
display(levels.query("DistanceLevel == 'similar'"))

print("Pares intermediários:")
display(levels.query("DistanceLevel == 'intermediate'"))

print("Pares distintos:")
display(levels.query("DistanceLevel == 'distinct'").sort_values("MahalanobisDistance", ascending=False))

#### Heatmap and Dendrogram

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/plots/attack_distance_dendrogram.png")

SimilarityDendrogramPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Dendrograma salvo em:", outputPath)

In [ ]:
distanceMatrixPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/reports/mahalanobis_distance_matrix.csv")
outputPath = Path("output/Similarity/CICDDoS2019_01-12_Attack_Similarity/analysis/stream/plots/mahalanobis_distance_heatmap.png")

SimilarityDistanceHeatmapPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Heatmap salvo em:", outputPath)